# CALVIN Automaton Sample-And-Rank

This notebook mirrors the toy-squares automaton-guidance notebook, but keeps the Calvin rollout / behavior-classification structure from `env_playground.ipynb`. It only runs unguided rollout and sample-and-rank; no gradient guidance.


In [ ]:
from pathlib import Path
from io import BytesIO
import importlib
import json
import os
import random
import sys
import time
import uuid

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pybullet as p
import torch
from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "calvin_experiments" / "calvin_rollout_utils.py").exists():
            return path
    raise FileNotFoundError(f"Could not find guided-diffusion repo root from {start}")


REPO_ROOT = find_repo_root()
for path in [REPO_ROOT, REPO_ROOT / "robomimic", REPO_ROOT / "calvin" / "calvin_env", REPO_ROOT / "calvin_experiments"]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

import robomimic.envs
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils
import robomimic.utils.python_utils as PyUtils
import robomimic.utils.torch_utils as TorchUtils

import calvin_experiments.calvin_rollout_utils as CalvinRolloutUtils
from calvin_experiments.label_calvin_world_model import LABEL_NAMES, label_scene_states
from calvin_experiments.train_automaton_world_model import AutomatonMLP

CalvinRolloutUtils = importlib.reload(CalvinRolloutUtils)
from calvin_experiments.calvin_rollout_utils import articulated_binaries_from_start_state, check_state_difference, classify_behavior


def seed_everything(seed):
    if seed is None:
        return
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


CONFIG_DIR = REPO_ROOT / "calvin_experiments" / "configs"
SCENE_CONFIG_PATH = CONFIG_DIR / "blocks_hidden.json"
VISUALIZATION_CONFIG_PATH = CONFIG_DIR / "visualization_freiburg_style.json"
CHECKPOINT_PATH = REPO_ROOT / "outputs/calvin/base_policy/calvin_D_base_dp/20260501015147/models/model_epoch_280.pth"
OUTPUT_ROOT = REPO_ROOT / "outputs/calvin/guidance_test"
VIDEO_FPS = 30

# The path from the prompt currently contains a toy-squares checkpoint (10D state, 4 labels).
USER_SUPPLIED_AUTOMATON_CKPT_PATH = REPO_ROOT / "outputs/automaton_world_model/training-run_2026-04-28_23-57-12/best_model.pt"
AUTOMATON_CKPT_PATH = REPO_ROOT / "outputs/calvin/automaton_world_model/h8_sh64_ah96_lh8_hh96_lr0.0001_epochs80_2026-05-01_04-25-33"

SCENE_INDEX = {
    "sliding_door": 0,
    "drawer": 1,
    "button": 2,
    "switch": 3,
    "lightbulb": 4,
    "green_light": 5,
}
BLOCK_POSE_SLICES = {
    "block_red": (6, 9, 9, 12),
    "block_blue": (12, 15, 15, 18),
    "block_pink": (18, 21, 21, 24),
}


def _to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def _format_onehot(values):
    return "[" + " ".join(f"{int(v):1d}" for v in values) + "]"


def _format_probs(values):
    return "[" + " ".join(f"{float(v):6.3f}" for v in values) + "]"


def automaton_model_for_eval(model_or_run_path, device):
    model_or_run_path = Path(model_or_run_path)
    ckpt_path = model_or_run_path / "best_model.pt" if model_or_run_path.is_dir() else model_or_run_path
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Automaton checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model_config = dict(ckpt.get("model_config", {}))
    if not model_config:
        state_dict = ckpt["model_state_dict"]
        model_config = {
            "state_dim": int(state_dict["state_enc.0.weight"].shape[1]),
            "label_dim": int(state_dict["label_enc.0.weight"].shape[1]),
            "action_chunk_dim": int(state_dict["action_enc.0.weight"].shape[1]),
            "state_hidden": int(state_dict["state_enc.0.weight"].shape[0]),
            "label_hidden": int(state_dict["label_enc.0.weight"].shape[0]),
            "action_hidden": int(state_dict["action_enc.0.weight"].shape[0]),
            "head_hidden": int(state_dict["head.0.weight"].shape[0]),
            "dropout": 0.0,
        }

    stats = ckpt.get("normalization_stats")
    if stats is None:
        stats_path = ckpt_path.parent / "normalization_stats.npz"
        if not stats_path.exists():
            raise FileNotFoundError(f"No normalization stats in checkpoint or at {stats_path}")
        z = np.load(stats_path)
        stats = {key: z[key] for key in z.files}
    stats = {key: np.asarray(value, dtype=np.float32) for key, value in stats.items()}

    model = AutomatonMLP(**model_config).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    meta = {
        "ckpt_path": str(ckpt_path),
        "run_dir": str(ckpt_path.parent),
        "model_config": model_config,
        "label_names": ckpt.get("label_names", LABEL_NAMES),
    }
    return model, stats, meta


if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Policy checkpoint not found: {CHECKPOINT_PATH}")
if not AUTOMATON_CKPT_PATH.exists():
    raise FileNotFoundError(f"Automaton checkpoint not found: {AUTOMATON_CKPT_PATH}")

device = TorchUtils.get_torch_device(try_to_use_cuda=True)
policy, ckpt_dict = FileUtils.policy_from_checkpoint(ckpt_path=str(CHECKPOINT_PATH), device=device, verbose=False)
env, _ = FileUtils.env_from_checkpoint(ckpt_dict=ckpt_dict, render=False, render_offscreen=True, verbose=False)
base_env_state = {key: np.asarray(value, dtype=np.float32).copy() for key, value in env.get_state().items()}

automaton_model, automaton_stats, automaton_meta = automaton_model_for_eval(AUTOMATON_CKPT_PATH, device)
print("Loaded policy:", CHECKPOINT_PATH)
print("Loaded automaton:", automaton_meta["ckpt_path"])
print("Automaton config:", automaton_meta["model_config"])
print("Label order:", list(LABEL_NAMES))
print("Target index mapping: 0=switch_on, 1=button_on, 2=drawer_open")
print("Device:", device)


In [ ]:
def get_calvin_unwrapped_env(env):
    current = env
    while hasattr(current, "env"):
        child = current.env
        if hasattr(child, "unwrapped"):
            return child.unwrapped
        current = child
    raise AttributeError("Could not find underlying CALVIN env")


def reset_env_to_scene_robot(env, scene, robot):
    state = {"scene": np.asarray(scene, dtype=np.float32).copy(), "robot": np.asarray(robot, dtype=np.float32).copy()}
    robomimic_env = env
    frame_stack_env = env if hasattr(env, "_get_initial_obs_history") else None
    if frame_stack_env is not None:
        robomimic_env = frame_stack_env.env
    gym_env = getattr(robomimic_env, "env", None)
    if gym_env is not None and hasattr(gym_env, "reset") and hasattr(robomimic_env, "get_observation"):
        raw_obs = gym_env.reset(scene_obs=state["scene"], robot_obs=state["robot"])
        if isinstance(raw_obs, tuple):
            raw_obs = raw_obs[0]
        robomimic_env._current_obs = raw_obs
        robomimic_env._current_reward = None
        robomimic_env._current_done = None
        obs = robomimic_env.get_observation(raw_obs)
        if frame_stack_env is not None:
            frame_stack_env.timestep = 0
            frame_stack_env.update_obs(obs, reset=True)
            frame_stack_env.obs_history = frame_stack_env._get_initial_obs_history(init_obs=obs)
            return frame_stack_env._get_stacked_obs_from_history()
        return obs
    return env.reset_to(state)


def render_visual_camera(env, video_cfg):
    if video_cfg.get("perspective") == "third_person":
        return env.render(
            mode="rgb_array",
            height=int(video_cfg["height"]),
            width=int(video_cfg["width"]),
        )

    calvin_env = get_calvin_unwrapped_env(env)
    width = int(video_cfg["width"])
    height = int(video_cfg["height"])
    view_matrix = p.computeViewMatrix(
        cameraEyePosition=video_cfg["look_from"],
        cameraTargetPosition=video_cfg["look_at"],
        cameraUpVector=video_cfg["up_vector"],
    )
    projection_matrix = p.computeProjectionMatrixFOV(
        fov=float(video_cfg["fov"]),
        aspect=width / height,
        nearVal=float(video_cfg["nearval"]),
        farVal=float(video_cfg["farval"]),
    )
    image = p.getCameraImage(
        width=width,
        height=height,
        viewMatrix=view_matrix,
        projectionMatrix=projection_matrix,
        physicsClientId=calvin_env.cid,
    )
    return np.reshape(image[2], (height, width, 4))[:, :, :3].astype(np.uint8)


def apply_block_scene_overrides(scene, blocks_cfg):
    for name, pose_cfg in blocks_cfg.get("poses", {}).items():
        pos_start, pos_end, rot_start, rot_end = BLOCK_POSE_SLICES[name]
        position = pose_cfg.get("position") if isinstance(pose_cfg, dict) else pose_cfg
        rotation = pose_cfg.get("rotation", [0.0, 0.0, 0.0]) if isinstance(pose_cfg, dict) else [0.0, 0.0, 0.0]
        scene[pos_start:pos_end] = np.asarray(position, dtype=np.float32)
        scene[rot_start:rot_end] = np.asarray(rotation, dtype=np.float32)


def fixed_scene_robot_from_config(scene_config_path=SCENE_CONFIG_PATH):
    with open(scene_config_path, "r") as f:
        scene_cfg = json.load(f)
    fixed_scene = np.asarray(base_env_state["scene"], dtype=np.float32).copy()
    fixed_robot = np.asarray(base_env_state["robot"], dtype=np.float32).copy()
    for name, value in scene_cfg.get("env_setup", {}).items():
        fixed_scene[SCENE_INDEX[name]] = float(value)
    apply_block_scene_overrides(fixed_scene, scene_cfg.get("blocks", {}))
    return fixed_scene, fixed_robot, scene_cfg


def _pose_from_joint_object(obj):
    link_state = obj.p.getLinkState(obj.uid, obj.joint_index, physicsClientId=obj.cid)
    return np.asarray(link_state[4], dtype=np.float32), np.asarray(link_state[5], dtype=np.float32)


def _pose_from_link_object(obj, link_index):
    link_state = obj.p.getLinkState(obj.uid, link_index, physicsClientId=obj.cid)
    return np.asarray(link_state[4], dtype=np.float32), np.asarray(link_state[5], dtype=np.float32)


def _aabb_from_object(obj, link_index=None):
    if link_index is None:
        lower, upper = obj.p.getAABB(obj.uid, physicsClientId=obj.cid)
    else:
        lower, upper = obj.p.getAABB(obj.uid, link_index, physicsClientId=obj.cid)
    return np.asarray(lower, dtype=np.float32), np.asarray(upper, dtype=np.float32)


def capture_scene_snapshot(env):
    scene = get_calvin_unwrapped_env(env).scene
    snapshot = {"controls": []}
    for obj in scene.buttons:
        lower, upper = _aabb_from_object(obj, obj.joint_index)
        snapshot["controls"].append({"kind": "button", "name": obj.name, "aabb": [lower.tolist(), upper.tolist()], "state": float(obj.get_state())})
    for obj in scene.doors:
        lower, upper = _aabb_from_object(obj, obj.joint_index)
        kind = "drawer" if "drawer" in obj.name else "slider"
        snapshot["controls"].append({"kind": kind, "name": obj.name, "aabb": [lower.tolist(), upper.tolist()], "state": float(obj.get_state())})
    for obj in scene.switches:
        lower, upper = _aabb_from_object(obj, obj.joint_index)
        snapshot["controls"].append({"kind": "switch", "name": obj.name, "aabb": [lower.tolist(), upper.tolist()], "state": float(obj.get_state())})
    for obj in scene.lights:
        lower, upper = _aabb_from_object(obj, obj.link_id)
        snapshot["controls"].append({"kind": "light", "name": obj.name, "aabb": [lower.tolist(), upper.tolist()], "state": float(obj.get_state())})
    return snapshot


def _rect_from_aabb(ax, lower, upper, **kwargs):
    ax.add_patch(Rectangle((float(lower[0]), float(lower[1])), float(upper[0] - lower[0]), float(upper[1] - lower[1]), **kwargs))


def _circle_from_aabb(ax, lower, upper, **kwargs):
    center = ((float(lower[0]) + float(upper[0])) / 2, (float(lower[1]) + float(upper[1])) / 2)
    radius = 0.5 * min(float(upper[0] - lower[0]), float(upper[1] - lower[1]))
    ax.add_patch(Circle(center, radius=radius, **kwargs))


def draw_scene_snapshot(ax, snapshot):
    styles = {
        "button": {"facecolor": "#111111", "edgecolor": "#000000", "alpha": 0.95},
        "slider": {"facecolor": "#4c566a", "edgecolor": "#1f2937", "alpha": 0.92},
        "drawer": {"facecolor": "#7a5c4d", "edgecolor": "#2f241f", "alpha": 0.9},
        "switch": {"facecolor": "#6b7280", "edgecolor": "black", "alpha": 0.9},
        "light": {"facecolor": "#9ca3af", "edgecolor": "black", "alpha": 0.75},
    }
    for control in snapshot.get("controls", []):
        lower = np.asarray(control["aabb"][0], dtype=np.float32)
        upper = np.asarray(control["aabb"][1], dtype=np.float32)
        style = styles.get(control["kind"], {"facecolor": "#adb5bd", "edgecolor": "black", "alpha": 0.8})
        patch_kwargs = dict(facecolor=style["facecolor"], edgecolor=style["edgecolor"], linewidth=1.0, alpha=style["alpha"])
        if control["kind"] == "button":
            _circle_from_aabb(ax, lower, upper, **patch_kwargs)
        else:
            _rect_from_aabb(ax, lower, upper, **patch_kwargs)
        ax.text(float((lower[0] + upper[0]) / 2), float(upper[1] + 0.01), control["name"].replace("base__", ""), ha="center", va="bottom", fontsize=7, color="black")
    ax.set_aspect("equal")
    ax.grid(color="white", alpha=0.2, linewidth=0.8)


def _scene_limits_from_snapshot(snapshot, eef_xy, padding=0.05):
    boxes = [(np.asarray(item["aabb"][0]), np.asarray(item["aabb"][1])) for item in snapshot.get("controls", [])]
    lower_xy = np.min([box[0][:2] for box in boxes], axis=0)
    upper_xy = np.max([box[1][:2] for box in boxes], axis=0)
    lower_xy = np.minimum(lower_xy, np.min(eef_xy[:, :2], axis=0))
    upper_xy = np.maximum(upper_xy, np.max(eef_xy[:, :2], axis=0))
    return (float(lower_xy[0] - padding), float(upper_xy[0] + padding)), (float(lower_xy[1] - padding), float(upper_xy[1] + padding))


def plot_rollout_xy(rollouts, scene_snapshot, title, save_path=None):
    paths = [np.asarray(item["eef_xy"], dtype=np.float32) for item in rollouts]
    behaviors = [item["behavior"] for item in rollouts]
    unique_behaviors = sorted(set(behaviors))
    cmap = plt.get_cmap("tab10")
    behavior_colors = {behavior: cmap(idx % cmap.N) for idx, behavior in enumerate(unique_behaviors)}
    all_eef_xy = np.concatenate(paths, axis=0)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_facecolor("#fbf7ef")
    draw_scene_snapshot(ax, scene_snapshot)
    for item, eef_xy in zip(rollouts, paths):
        color = behavior_colors[item["behavior"]]
        ax.plot(eef_xy[:, 0], eef_xy[:, 1], color=color, linewidth=2.0, alpha=0.75)
        ax.scatter(eef_xy[0, 0], eef_xy[0, 1], c=[color], s=28, edgecolors="white", linewidths=0.7, alpha=0.85)
        ax.scatter(eef_xy[-1, 0], eef_xy[-1, 1], c=[color], s=42, edgecolors="black", linewidths=0.7, alpha=0.9)

    xlim, ylim = _scene_limits_from_snapshot(scene_snapshot, all_eef_xy)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.set_xlabel("world x [m]")
    ax.set_ylabel("world y [m]")
    ax.legend(
        handles=[Line2D([0], [0], color=behavior_colors[b], lw=2.5, label=f"{b} ({behaviors.count(b)})") for b in unique_behaviors],
        loc="upper right",
    )
    png = BytesIO()
    fig.savefig(png, format="png", dpi=100, bbox_inches="tight")
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print("plot:", save_path)
    plt.close(fig)
    display(Image(data=png.getvalue()))


In [ ]:
def current_automaton_state_and_label(env):
    state = env.get_state()
    robot = np.asarray(state["robot"], dtype=np.float32).reshape(-1)
    scene = np.asarray(state["scene"], dtype=np.float32).reshape(-1)
    automaton_state = np.concatenate([robot, scene]).astype(np.float32)
    automaton_label = label_scene_states(scene[None, :])[0].astype(np.float32)
    return automaton_state, automaton_label


def _load_video_cfg():
    with open(VISUALIZATION_CONFIG_PATH, "r") as f:
        return json.load(f)


def _save_rollout_artifacts(rollout, frames, output_dir, rollout_tag, video_cfg, fps=VIDEO_FPS):
    rollout_dir = Path(output_dir) / rollout_tag
    rollout_dir.mkdir(parents=True, exist_ok=True)
    video_path = rollout_dir / f"{rollout['scene_config']}_{rollout_tag}_{video_cfg['perspective']}.mp4"
    trace_path = rollout_dir / "rollout_trace.npz"
    scene_snapshot_path = rollout_dir / "scene_snapshot.json"

    imageio.mimsave(video_path, frames, fps=int(fps))
    np.savez_compressed(
        trace_path,
        actions=rollout["actions"],
        rewards=rollout["rewards"],
        dones=rollout["dones"],
        scene_states=rollout["scene_states"],
        robot_states=rollout["robot_states"],
        eef_xy=rollout["eef_xy"],
        detected_behavior=np.asarray(rollout["behavior"]),
        detected_behavior_step=np.asarray(rollout["behavior_step"], dtype=np.int32),
        termination_step=np.asarray(rollout["termination_step"], dtype=np.int32),
        termination_reason=np.asarray(rollout["termination_reason"]),
        rollout_seed=np.asarray(rollout["seed"], dtype=np.int32),
        initial_label=np.asarray(rollout["initial_label"], dtype=np.int32),
        final_label=np.asarray(rollout["final_label"], dtype=np.int32),
        scene_config=np.asarray(rollout["scene_config"]),
    )
    with open(scene_snapshot_path, "w") as f:
        json.dump(rollout["scene_snapshot"], f, indent=2)

    rollout["video"] = video_path
    rollout["trace"] = trace_path
    rollout["scene_snapshot_path"] = scene_snapshot_path
    rollout["rollout_dir"] = rollout_dir
    return rollout


def rollout_policy_once(
    seed=2,
    horizon=100,
    stop_on_behavior=True,
    action_provider=None,
    output_dir=None,
    rollout_tag=None,
    save_video=True,
    video_cfg=None,
):
    fixed_scene, fixed_robot, scene_cfg = fixed_scene_robot_from_config()
    seed_everything(seed)
    obs = reset_env_to_scene_robot(env, fixed_scene, fixed_robot)
    scene_snapshot = capture_scene_snapshot(env)
    policy.start_episode()

    should_save = output_dir is not None and bool(save_video)
    if should_save and video_cfg is None:
        video_cfg = _load_video_cfg()
    frames = [render_visual_camera(env, video_cfg)] if should_save else []

    start_state = env.get_state()
    start_scene = np.asarray(start_state["scene"], dtype=np.float32).copy()
    binaries = articulated_binaries_from_start_state(start_scene)
    state0, label0 = current_automaton_state_and_label(env)

    actions, rewards, dones = [], [], []
    scene_states = [start_scene.copy()]
    robot_states = [np.asarray(start_state["robot"], dtype=np.float32).copy()]
    eef_xy = [robot_states[-1][:2].copy()]
    records = []

    detected_behavior = "none"
    detected_step = -1
    termination_reason = "horizon"
    action_queue = []

    for step in range(int(horizon)):
        if action_provider is None:
            action = policy(ob=obs)
        else:
            if not action_queue:
                new_actions, record = action_provider(obs, env, step)
                action_queue.extend(np.asarray(new_actions, dtype=np.float32))
                records.append(record)
            action = action_queue.pop(0)

        action_to_step = np.asarray(action, dtype=np.float32).copy()
        actions.append(action_to_step.copy())
        obs, reward, done, info = env.step(action_to_step)
        state = env.get_state()
        scene = np.asarray(state["scene"], dtype=np.float32).copy()
        robot = np.asarray(state["robot"], dtype=np.float32).copy()

        rewards.append(float(reward))
        dones.append(bool(done))
        scene_states.append(scene)
        robot_states.append(robot)
        eef_xy.append(robot[:2].copy())
        if should_save:
            frames.append(render_visual_camera(env, video_cfg))

        triggered = check_state_difference(start_scene, scene, robot[:3], binaries, for_display=False)
        if detected_step < 0 and triggered:
            detected_behavior = classify_behavior(start_scene, scene, robot[:3], binaries, for_display=False)
            detected_step = step + 1
            if stop_on_behavior:
                termination_reason = "behavior"
                break
        if done:
            termination_reason = "env_done"
            break

    statef, labelf = current_automaton_state_and_label(env)
    rollout = {
        "scene_config": scene_cfg["name"],
        "seed": int(seed),
        "behavior": detected_behavior,
        "behavior_step": int(detected_step),
        "termination_step": len(actions),
        "termination_reason": termination_reason,
        "return": float(np.sum(rewards)),
        "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "dones": np.asarray(dones, dtype=bool),
        "scene_states": np.asarray(scene_states, dtype=np.float32),
        "robot_states": np.asarray(robot_states, dtype=np.float32),
        "eef_xy": np.asarray(eef_xy, dtype=np.float32),
        "initial_label": label0.astype(int).tolist(),
        "final_label": labelf.astype(int).tolist(),
        "records": records,
        "scene_snapshot": scene_snapshot,
    }
    if should_save:
        if rollout_tag is None:
            rollout_tag = f"rollout_seed_{int(seed):03d}"
        _save_rollout_artifacts(rollout, frames, output_dir, rollout_tag, video_cfg)
    return rollout


UNGUIDED_SEED = 2
UNGUIDED_HORIZON = 100
unguided_output_dir = OUTPUT_ROOT / f"unguided_seed{UNGUIDED_SEED}_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
unguided_rollout = rollout_policy_once(
    seed=UNGUIDED_SEED,
    horizon=UNGUIDED_HORIZON,
    stop_on_behavior=True,
    output_dir=unguided_output_dir,
    rollout_tag="rollout_000",
)
print(
    f"Unguided rollout seed={UNGUIDED_SEED}: behavior={unguided_rollout['behavior']} "
    f"@ {unguided_rollout['behavior_step']}, steps={unguided_rollout['termination_step']} "
    f"({unguided_rollout['termination_reason']}), return={unguided_rollout['return']:.3f}"
)
print("initial automaton label:", _format_onehot(unguided_rollout["initial_label"]), "final:", _format_onehot(unguided_rollout["final_label"]))
print("video:", unguided_rollout.get("video"))
print("trace:", unguided_rollout.get("trace"))
unguided_plot_path = unguided_output_dir / "rollout_xy.png"
plot_rollout_xy(
    [unguided_rollout],
    unguided_rollout["scene_snapshot"],
    f"Unguided rollout | {unguided_rollout['scene_config']} | seed {UNGUIDED_SEED}",
    save_path=unguided_plot_path,
)


In [ ]:
def _repeat_obs_batch(obs_tensor, n):
    if n == 1:
        return obs_tensor
    return {key: value.repeat((n,) + (1,) * (value.ndim - 1)) for key, value in obs_tensor.items()}


def _unnormalize_action_sequence(action_sequence):
    action_np = _to_numpy(action_sequence).astype(np.float32)
    if policy.action_normalization_stats is None:
        return action_np

    original_shape = action_np.shape
    flat_actions = action_np.reshape(-1, original_shape[-1])
    action_keys = policy.policy.global_config.train.action_keys
    action_shapes = {key: policy.action_normalization_stats[key]["offset"].shape[1:] for key in policy.action_normalization_stats}
    action_dict = PyUtils.vector_to_action_dict(flat_actions, action_shapes=action_shapes, action_keys=action_keys)
    action_dict = ObsUtils.unnormalize_dict(action_dict, normalization_stats=policy.action_normalization_stats)
    return PyUtils.action_dict_to_vector(action_dict, action_keys=action_keys).reshape(original_shape)


def predict_future_label_probs(automaton_state, automaton_label, action_chunks):
    action_chunks = np.asarray(action_chunks, dtype=np.float32)
    if action_chunks.ndim != 3:
        raise ValueError(f"Expected action_chunks shape (N, H, A), got {action_chunks.shape}")

    n_candidates, _, action_dim = action_chunks.shape
    state_dim = len(automaton_stats["states_mean"])
    action_chunk_dim = len(automaton_stats["actions_mean"])
    if len(automaton_state) != state_dim:
        raise ValueError(
            f"Automaton state dim mismatch: rollout has {len(automaton_state)}, checkpoint expects {state_dim}. "
            f"If this mentions 10 vs 39, the checkpoint is the toy-squares model: {USER_SUPPLIED_AUTOMATON_CKPT_PATH}"
        )
    if action_chunk_dim % action_dim != 0:
        raise ValueError(f"Automaton action chunk dim {action_chunk_dim} is not divisible by action dim {action_dim}")
    automaton_horizon = action_chunk_dim // action_dim
    if action_chunks.shape[1] < automaton_horizon:
        raise ValueError(f"Policy produced {action_chunks.shape[1]} actions, automaton expects horizon {automaton_horizon}")

    scored_chunks = action_chunks[:, :automaton_horizon, :].reshape(n_candidates, -1)
    states = np.repeat(np.asarray(automaton_state, dtype=np.float32)[None, :], n_candidates, axis=0)
    labels = np.repeat(np.asarray(automaton_label, dtype=np.float32)[None, :], n_candidates, axis=0)

    states_t = torch.as_tensor((states - automaton_stats["states_mean"]) / automaton_stats["states_std"], device=device, dtype=torch.float32)
    actions_t = torch.as_tensor((scored_chunks - automaton_stats["actions_mean"]) / automaton_stats["actions_std"], device=device, dtype=torch.float32)
    labels_t = torch.as_tensor(labels, device=device, dtype=torch.float32)
    with torch.no_grad():
        return torch.sigmoid(automaton_model(states_t, actions_t, labels_t)).detach().cpu().numpy(), automaton_horizon


def make_sample_rank_action_provider(target_label_idx, n_candidates):
    target_label_idx = int(target_label_idx)
    n_candidates = int(n_candidates)
    if target_label_idx < 0 or target_label_idx >= len(LABEL_NAMES):
        raise ValueError(f"target_label_idx must be in [0, {len(LABEL_NAMES) - 1}], got {target_label_idx}")
    if n_candidates < 1:
        raise ValueError("n_candidates must be >= 1")

    def action_provider(obs, env, step):
        automaton_state, automaton_label = current_automaton_state_and_label(env)
        obs_tensor = policy._prepare_observation(obs)
        obs_tensor_rank = _repeat_obs_batch(obs_tensor, n_candidates)
        with torch.no_grad():
            action_chunk_n = policy.policy._get_action_trajectory(obs_dict=obs_tensor_rank)
        action_chunk = _unnormalize_action_sequence(action_chunk_n)
        candidate_probs, automaton_horizon = predict_future_label_probs(automaton_state, automaton_label, action_chunk)
        candidate_scores = candidate_probs[:, target_label_idx]
        selected_idx = int(np.argmax(candidate_scores))
        record = {
            "t": int(step),
            "target_label_idx": target_label_idx,
            "target_label_name": LABEL_NAMES[target_label_idx],
            "current_label": automaton_label.astype(int).tolist(),
            "selected_idx": selected_idx,
            "selected_score": float(candidate_scores[selected_idx]),
            "pred_probs": candidate_probs[selected_idx].tolist(),
            "candidate_scores": candidate_scores.tolist(),
        }
        return action_chunk[selected_idx, :automaton_horizon, :], record

    return action_provider


# Target labels from label_calvin_world_model.py: 0=switch_on, 1=button_on, 2=drawer_open.
TARGET_LABEL_IDX = 1
SAMPLE_AND_RANK_N = 1
SAMPLE_AND_RANK_SEED = 2
SAMPLE_AND_RANK_HORIZON = 100

sample_rank_output_dir = OUTPUT_ROOT / f"sample_rank_target{TARGET_LABEL_IDX}_seed{SAMPLE_AND_RANK_SEED}_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
sample_rank_action_provider = make_sample_rank_action_provider(TARGET_LABEL_IDX, SAMPLE_AND_RANK_N)
sample_rank_rollout = rollout_policy_once(
    seed=SAMPLE_AND_RANK_SEED,
    horizon=SAMPLE_AND_RANK_HORIZON,
    stop_on_behavior=True,
    action_provider=sample_rank_action_provider,
    output_dir=sample_rank_output_dir,
    rollout_tag="rollout_000",
)

print(f"Sample-and-rank target={TARGET_LABEL_IDX} ({LABEL_NAMES[TARGET_LABEL_IDX]}), N={SAMPLE_AND_RANK_N}")
print(
    f"Rollout: behavior={sample_rank_rollout['behavior']} @ {sample_rank_rollout['behavior_step']}, "
    f"steps={sample_rank_rollout['termination_step']} ({sample_rank_rollout['termination_reason']}), "
    f"return={sample_rank_rollout['return']:.3f}, final_label={_format_onehot(sample_rank_rollout['final_label'])}"
)
print("video:", sample_rank_rollout.get("video"))
print("trace:", sample_rank_rollout.get("trace"))
print(f"{'t':>4}  {'pick':>4}  {'score':>7}  {'current':>9}  {'pred_future_prob':>27}")
for record in sample_rank_rollout["records"]:
    print(
        f"{record['t']:4d}  {record['selected_idx']:4d}  {record['selected_score']:7.3f}  "
        f"{_format_onehot(record['current_label']):>9}  {_format_probs(record['pred_probs']):>27}"
    )
sample_rank_plot_path = sample_rank_output_dir / "rollout_xy.png"
plot_rollout_xy(
    [sample_rank_rollout],
    sample_rank_rollout["scene_snapshot"],
    f"Sample-and-rank -> {LABEL_NAMES[TARGET_LABEL_IDX]} | seed {SAMPLE_AND_RANK_SEED}",
    save_path=sample_rank_plot_path,
)


In [ ]:
from collections import Counter


def run_sample_rank_batch(target_label_idx, n_rollouts=20, seed_start=0, seeds=None, n_candidates=16, horizon=100):
    target_label_idx = int(target_label_idx)
    if seeds is None:
        seeds = list(range(int(seed_start), int(seed_start) + int(n_rollouts)))
    else:
        seeds = [int(seed) for seed in seeds]

    run_dir = OUTPUT_ROOT / (
        f"sample_rank_batch_target{target_label_idx}_n{len(seeds)}_candidates{int(n_candidates)}_"
        f"{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)
    video_cfg = _load_video_cfg()

    action_provider = make_sample_rank_action_provider(target_label_idx, n_candidates)
    rollouts = []
    for rollout_idx, seed in enumerate(seeds):
        rollout = rollout_policy_once(
            seed=seed,
            horizon=horizon,
            stop_on_behavior=True,
            action_provider=action_provider,
            output_dir=run_dir,
            rollout_tag=f"rollout_{rollout_idx:03d}_seed_{seed:03d}",
            video_cfg=video_cfg,
        )
        rollouts.append(rollout)
        print(
            f"seed {seed:03d} -> behavior={rollout['behavior']:>12} "
            f"@ {rollout['behavior_step']:>3}, steps={rollout['termination_step']:>3}, "
            f"final_label={_format_onehot(rollout['final_label'])}, video={rollout['video']}"
        )

    behavior_counts = Counter(rollout["behavior"] for rollout in rollouts)
    final_label_counts = Counter(tuple(rollout["final_label"]) for rollout in rollouts)

    print("\nBehavior split:")
    for behavior, count in behavior_counts.most_common():
        print(f"  {behavior:>12}: {count:>3}/{len(rollouts)} ({count / len(rollouts):.1%})")

    print("\nFinal label split:")
    for label, count in final_label_counts.most_common():
        print(f"  {_format_onehot(label)}: {count:>3}/{len(rollouts)} ({count / len(rollouts):.1%})")

    summary = {
        "target_label_idx": target_label_idx,
        "target_label_name": LABEL_NAMES[target_label_idx],
        "n_rollouts": len(rollouts),
        "n_candidates": int(n_candidates),
        "horizon": int(horizon),
        "seeds": seeds,
        "behavior_counts": dict(behavior_counts),
        "final_label_counts": {str(label): count for label, count in final_label_counts.items()},
        "rollouts": [
            {
                "seed": rollout["seed"],
                "behavior": rollout["behavior"],
                "behavior_step": rollout["behavior_step"],
                "termination_step": rollout["termination_step"],
                "termination_reason": rollout["termination_reason"],
                "return": rollout["return"],
                "initial_label": rollout["initial_label"],
                "final_label": rollout["final_label"],
                "video": str(rollout["video"]),
                "trace": str(rollout["trace"]),
            }
            for rollout in rollouts
        ],
    }
    summary_path = run_dir / "batch_summary.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    batch_plot_path = run_dir / "batch_rollout_xy.png"
    plot_rollout_xy(
        rollouts,
        rollouts[0]["scene_snapshot"],
        f"Sample-and-rank batch -> {LABEL_NAMES[target_label_idx]} | N={len(rollouts)}, candidates={n_candidates}",
        save_path=batch_plot_path,
    )
    print("\nOutput dir:", run_dir)
    print("Summary JSON:", summary_path)
    return rollouts, behavior_counts, final_label_counts, run_dir


# Target labels from label_calvin_world_model.py: 0=switch_on, 1=button_on, 2=drawer_open.
BATCH_TARGET_LABEL_IDX = 2
BATCH_N_ROLLOUTS = 10
BATCH_SEED_START = 0
BATCH_SEEDS = None  # Optional: set e.g. [0, 3, 7, 11] instead of using BATCH_SEED_START/BATCH_N_ROLLOUTS.
BATCH_SAMPLE_AND_RANK_N = 1
BATCH_HORIZON = 100

batch_sample_rank_rollouts, batch_behavior_counts, batch_final_label_counts, batch_output_dir = run_sample_rank_batch(
    target_label_idx=BATCH_TARGET_LABEL_IDX,
    n_rollouts=BATCH_N_ROLLOUTS,
    seed_start=BATCH_SEED_START,
    seeds=BATCH_SEEDS,
    n_candidates=BATCH_SAMPLE_AND_RANK_N,
    horizon=BATCH_HORIZON,
)
